In [1]:
%pip install google-generativeai python-dotenv

  Using cached google_generativeai-0.8.6-py3-none-any.whl.metadata (3.9 kB)
  Using cached google_ai_generativelanguage-0.6.15-py3-none-any.whl.metadata (5.7 kB)
  Using cached google_api_core-2.31.0-py3-none-any.whl.metadata (3.2 kB)
  Using cached google_api_python_client-2.197.0-py3-none-any.whl.metadata (7.0 kB)
  Using cached google_auth-2.55.0-py3-none-any.whl.metadata (5.1 kB)
  Using cached proto_plus-1.28.0-py3-none-any.whl.metadata (2.2 kB)
  Using cached googleapis_common_protos-1.75.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached grpcio_status-1.81.1-py3-none-any.whl.metadata (1.2 kB)
  Using cached cryptography-49.0.0-cp311-abi3-win_amd64.whl.metadata (4.3 kB)
INFO: pip is looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
  Using cached grpcio_status-1.81.0-py3-none-any.whl.metadata (1.2 kB)
  Using cached grpcio_status-1.80.0-py3-none-any.whl.metadata (1.3 kB)
  Using cached grpcio_

In [3]:
import json
try:
        with open("queries/scifact_cs.jsonl", encoding="utf-8") as f:
            done = [json.loads(l) for l in f]
        print(f"scifact_cs.jsonl already has {len(done)} queries saved")
except FileNotFoundError:
        print("no scifact file saved yet")

no scifact file saved yet


In [4]:
%pip install groq

  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
Using cached sniffio-1.3.1-py3-none-any.whl (10 kB)

   ------------- -------------------------- 1/3 [distro]
   -------------------------- ------------- 2/3 [groq]
   -------------------------- ------------- 2/3 [groq]
   -------------------------- ------------- 2/3 [groq]
   -------------------------- ------------- 2/3 [groq]
   -------------------------- ------------- 2/3 [groq]
   -------------------------- ------------- 2/3 [groq]
   -------------------------- ------------- 2/3 [groq]
   -------------------------- ------------- 2/3 [groq]
   -------------------------- ------------- 2/3 [groq]
   -------------------------- ------------- 2/3 [groq]
   ---------------------------------------- 3/3 [groq]

Note: you may need to restart the kernel to use updated packages.


In [7]:
%pip install --force-reinstall --no-cache-dir pydantic


   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------- ----------------------------- 0.5/2.1 MB 2.1 MB/s eta 0:00:01
   ----------------------------------- ---- 1.8/2.1 MB 4.6 MB/s eta 0:00:01
   ---------------------------------------- 2.1/2.1 MB 5.0 MB/s  0:00:00

  Attempting uninstall: typing-extensions

    Found existing installation: typing_extensions 4.15.0

    Uninstalling typing_extensions-4.15.0:

   ---------------------------------------- 0/5 [typing-extensions]
   ---------------------------------------- 0/5 [typing-extensions]
   ---------------------------------------- 0/5 [typing-extensions]
   ---------------------------------------- 0/5 [typing-extensions]
   ---------------------------------------- 0/5 [typing-extensions]
      Successfully uninstalled typing_extensions-4.15.0
   ---------------------------------------- 0/5 [typing-extensions]
  Attempting uninstall: annotated-types
   ---------------------------------------- 0/5 [typing

  You can safely remove it manually.


In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
groq_key = os.getenv("GROQ_API_KEY")
print("Groq key loaded:", bool(groq_key), "| length:", len(groq_key) if groq_key else 0)

Groq key loaded: True | length: 56


In [2]:
from groq import Groq
import json, time
from beir.datasets.data_loader import GenericDataLoader

client = Groq(api_key=groq_key)
MODEL = "llama-3.3-70b-versatile"

BATCH_PROMPT = """You are a native Urdu speaker from Pakistan. For EACH numbered English search query below, rewrite it in TWO ways as a real Pakistani person would naturally type it:

1. "ur_en": Urdu-English code-switched using NATIVE URDU SCRIPT for Urdu words. Keep technical/proper-noun English terms in English. Natural code-mixing, NOT full translation.
2. "roman": Urdu-English code-switched in ROMAN URDU (Latin letters, how people type Urdu on phones). Keep technical/English terms in English.

Keep each MEANING identical to the original. Do not add or remove information.

Return ONLY a valid JSON array, one object per query, in the same order:
[{"n": 1, "ur_en": "...", "roman": "..."}, {"n": 2, "ur_en": "...", "roman": "..."}]

Queries:
"""

def transform_batch(items, max_retries=5):
    numbered = "\n".join(f"{i+1}. {txt}" for i, (qid, txt) in enumerate(items))
    for attempt in range(max_retries):
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                messages=[{"role": "user", "content": BATCH_PROMPT + numbered}],
                temperature=0.3,
            )
            raw = resp.choices[0].message.content.strip()
            raw = raw.replace("```json", "").replace("```", "").strip()
            start, end = raw.find("["), raw.rfind("]")
            arr = json.loads(raw[start:end+1])
            out = {}
            for obj in arr:
                qid = items[obj["n"] - 1][0]
                out[qid] = {"ur_en": obj["ur_en"], "roman": obj["roman"]}
            return out
        except Exception as e:
            print(f"  issue ({type(e).__name__}: {e}), retry {attempt+1}...")
            time.sleep(5)
    raise RuntimeError("batch failed after retries")

# 5-query quality test
corpus, queries, qrels = GenericDataLoader(data_folder="data/scifact").load(split="test")
test_items = [(qid, queries[qid]) for qid in list(queries.keys())[:5]]
test_out = transform_batch(test_items)
for qid, txt in test_items:
    print("EN   :", txt)
    print("UR-EN:", test_out[qid]["ur_en"])
    print("ROMAN:", test_out[qid]["roman"])
    print("---")

f:\Research Paper\AI2ML code-switching benchmark\venv\Lib\site-packages\beir\datasets\data_loader.py:8: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm
100%|██████████| 5183/5183 [00:00<00:00, 153730.84it/s]


EN   : 0-dimensional biomaterials show inductive properties.
UR-EN: 0 بعدی بائیو میٹریلز میں انڈکٹو پراپرٹیز دکھائی دیتی ہیں
ROMAN: 0 dimensonal bio materials main inductive properties dikhai deti hain
---
EN   : 1,000 genomes project enables mapping of genetic sequence variation consisting of rare variants with larger penetrance effects than common variants.
UR-EN: 1000 جینوم پروجیکٹ ریئر ویریئنٹس کے ساتھ جینیٹک سیکنس ویریئشن کے میپنگ کو ایبل کرتا ہے جو کہ کامن ویریئنٹس کے مقابلے میں بڑے پنٹرنس افیکٹس رکھتے ہیں
ROMAN: 1000 genome project rare variants ke saath genetic sequence variation ke mapping ko able karta hai jo ke common variants ke muqable main bade penetrance effects rakhte hain
---
EN   : 1/2000 in UK have abnormal PrP positivity.
UR-EN: یوکے میں 1/2000 افراد میں ایبノرمل PrP پوزٹیویٹی ہوتی ہے
ROMAN: yuk main 1/2000 afrad main abnormal prp positivity hoti hai
---
EN   : 5% of perinatal mortality is due to low birth weight.
UR-EN: پیرینٹل مورتالٹی کا 5% لو بِرтھ وےٹ کی وجہ سے 

In [3]:
BATCH_PROMPT = """You are a native Urdu speaker from Pakistan. For EACH numbered English query, rewrite it in TWO ways a real Pakistani would type it.

CRITICAL RULES:
- In "ur_en": write ONLY the Urdu/grammatical words in Urdu script. Keep ALL English technical terms, proper nouns, and abbreviations in their ORIGINAL ENGLISH (Latin) spelling. DO NOT transliterate English words into Urdu script.
- In "roman": write the whole thing in Latin letters (Roman Urdu), keeping English technical terms as English.
- Keep abbreviations/proper nouns EXACTLY as in the query (UK, PrP, B12, DNA, etc.).
- Use ONLY Urdu-script characters and English (Latin) letters. NEVER use any other alphabet (no Arabic-only, no Cyrillic).
- Keep the meaning identical. Do not add or remove information.

EXAMPLE:
English: What causes vitamin B12 deficiency in diabetes patients?
ur_en: diabetes patients میں vitamin B12 deficiency کس وجہ سے ہوتی ہے؟
roman: diabetes patients mein vitamin B12 deficiency kis wajah se hoti hai?

Return ONLY a valid JSON array, same order:
[{"n": 1, "ur_en": "...", "roman": "..."}, {"n": 2, "ur_en": "...", "roman": "..."}]

Queries:
"""

# re-test the same 5
test_out = transform_batch(test_items)
for qid, txt in test_items:
    print("EN   :", txt)
    print("UR-EN:", test_out[qid]["ur_en"])
    print("ROMAN:", test_out[qid]["roman"])
    print("---")

EN   : 0-dimensional biomaterials show inductive properties.
UR-EN: 0-dimensional biomaterials میں inductive properties ظاہر ہوتے ہیں
ROMAN: 0-dimensional biomaterials mein inductive properties zaahir hotey hain
---
EN   : 1,000 genomes project enables mapping of genetic sequence variation consisting of rare variants with larger penetrance effects than common variants.
UR-EN: 1,000 genomes project کی وجہ سے genetic sequence variation کا mapping ہو سکتا ہے جو rare variants پر مشتمل ہوتا ہے جو common variants کے مقابلے میں بڑے penetrance effects کا حامل ہوتا ہے
ROMAN: 1,000 genomes project ki wajah se genetic sequence variation ka mapping ho sakta hai jo rare variants par mushtamil hota hai jo common variants ke muqable mein bade penetrance effects ka hamil hota hai
---
EN   : 1/2000 in UK have abnormal PrP positivity.
UR-EN: UK میں 1/2000 افراد میں abnormal PrP positivity پائی جاتی ہے
ROMAN: UK mein 1/2000 afrad mein abnormal PrP positivity paaee jati hai
---
EN   : 5% of perinatal mort

In [4]:
import os, json, time
os.makedirs("queries", exist_ok=True)

def run_dataset(ds_name, batch_size=10, sleep_between=2):
    corpus, queries, qrels = GenericDataLoader(data_folder=f"data/{ds_name}").load(split="test")
    qids = list(queries.keys())
    items = [(qid, queries[qid]) for qid in qids]
    path = f"queries/{ds_name}_cs.jsonl"
    results, failed = {}, []
    n_batches = (len(items) + batch_size - 1) // batch_size

    with open(path, "w", encoding="utf-8") as f:        # incremental write
        for b in range(0, len(items), batch_size):
            batch = items[b:b+batch_size]
            bi = b // batch_size + 1
            try:
                out = transform_batch(batch)
                for qid, _ in batch:
                    if qid in out:
                        rec = {"id": qid, "EN": queries[qid],
                                "ur_en": out[qid]["ur_en"], "roman": out[qid]["roman"]}
                        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
                        f.flush()
                        results[qid] = out[qid]
                print(f"  {ds_name} batch {bi}/{n_batches} OK ({len(out)})")
            except Exception as e:
                print(f"  {ds_name} batch {bi}/{n_batches} FAILED: {e}")
                failed.extend([qid for qid, _ in batch])
            time.sleep(sleep_between)

    print(f"{ds_name}: saved {len(results)}/{len(qids)} -> {path} | failed: {len(failed)}")
    return failed

all_failed = {}
for ds in ["scifact", "nfcorpus"]:
    print(f"\n=== {ds} ===")
    all_failed[ds] = run_dataset(ds)

print("\nDone. Failed counts:", {k: len(v) for k, v in all_failed.items()})


=== scifact ===


100%|██████████| 5183/5183 [00:00<00:00, 155788.78it/s]


  scifact batch 1/30 OK (10)
  scifact batch 2/30 OK (10)
  scifact batch 3/30 OK (10)
  scifact batch 4/30 OK (10)
  issue (JSONDecodeError: Expecting ',' delimiter: line 11 column 148 (char 1485)), retry 1...
  issue (JSONDecodeError: Expecting ',' delimiter: line 11 column 148 (char 1479)), retry 2...
  scifact batch 5/30 OK (10)
  scifact batch 6/30 OK (10)
  scifact batch 7/30 OK (10)
  scifact batch 8/30 OK (10)
  scifact batch 9/30 OK (10)
  scifact batch 10/30 OK (10)
  scifact batch 11/30 OK (10)
  scifact batch 12/30 OK (10)
  scifact batch 13/30 OK (10)
  scifact batch 14/30 OK (10)
  scifact batch 15/30 OK (10)
  scifact batch 16/30 OK (10)
  scifact batch 17/30 OK (10)
  scifact batch 18/30 OK (10)
  scifact batch 19/30 OK (10)
  scifact batch 20/30 OK (10)
  scifact batch 21/30 OK (10)
  scifact batch 22/30 OK (10)
  scifact batch 23/30 OK (10)
  scifact batch 24/30 OK (10)
  scifact batch 25/30 OK (10)
  scifact batch 26/30 OK (10)
  issue (JSONDecodeError: Expecting ','

100%|██████████| 3633/3633 [00:00<00:00, 77292.50it/s]


  nfcorpus batch 1/33 OK (10)
  nfcorpus batch 2/33 OK (10)
  nfcorpus batch 3/33 OK (10)
  nfcorpus batch 4/33 OK (10)
  nfcorpus batch 5/33 OK (10)
  nfcorpus batch 6/33 OK (10)
  nfcorpus batch 7/33 OK (10)
  nfcorpus batch 8/33 OK (10)
  nfcorpus batch 9/33 OK (10)
  nfcorpus batch 10/33 OK (10)
  nfcorpus batch 11/33 OK (10)
  nfcorpus batch 12/33 OK (10)
  nfcorpus batch 13/33 OK (10)
  nfcorpus batch 14/33 OK (10)
  nfcorpus batch 15/33 OK (10)
  issue (JSONDecodeError: Expecting value: line 2 column 19 (char 20)), retry 1...
  issue (JSONDecodeError: Expecting value: line 2 column 19 (char 20)), retry 2...
  nfcorpus batch 16/33 OK (10)
  nfcorpus batch 17/33 OK (10)
  nfcorpus batch 18/33 OK (10)
  nfcorpus batch 19/33 OK (10)
  nfcorpus batch 20/33 OK (10)
  nfcorpus batch 21/33 OK (10)
  nfcorpus batch 22/33 OK (10)
  nfcorpus batch 23/33 OK (10)
  nfcorpus batch 24/33 OK (10)
  nfcorpus batch 25/33 OK (10)
  nfcorpus batch 26/33 OK (10)
  nfcorpus batch 27/33 OK (10)
  nfco